# latent_agentic_planning — real run on a GPU

Trains the latent Planner–Executor pipeline on a free GPU and produces the 3-way comparison
(Qwen executor / strategy prompt / planner+latent plan).

**First: Runtime → Change runtime type → T4 GPU.** Then Runtime → Run all.

In [ ]:
!git clone https://github.com/sinha-k-prat/latent_agentic_planning.git
%cd latent_agentic_planning
!pip -q install -r requirements.txt

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

## Train
0.5B trainee + 7B judge in 4-bit (fits a 16 GB T4). `steps=300` finishes in ~20–40 min;
bump to 1000 for the full run, or swap `model.base` to `Qwen/Qwen2.5-1.5B-Instruct`.
Planner weights are saved to `runs/default/checkpoints/{best,final}/`.

In [ ]:
!python -m src.train \
  model.base=Qwen/Qwen2.5-0.5B-Instruct \
  model.judge=Qwen/Qwen2.5-7B-Instruct judge.kind=local judge.load_in_4bit=true \
  plan.plan_mode=gumbel_codebook plan.N=16 plan.K=512 \
  train.steps=300 train.batch_size=8 train.group_size=8 train.max_new_tokens=256 \
  logging.eval_every=150 device=cuda

## Three-way comparison

In [ ]:
!python -m src.evaluate --ckpt runs/default/checkpoints/best --n 200 \
  model.base=Qwen/Qwen2.5-0.5B-Instruct \
  model.judge=Qwen/Qwen2.5-7B-Instruct judge.kind=local judge.load_in_4bit=true device=cuda
from IPython.display import Markdown
Markdown(open('results/comparison.md').read())

## Training curves (reward + gradient signal into the plan vectors)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv('runs/default/metrics.csv')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(df.step, df.reward_mean); ax[0].set_title('mean reward'); ax[0].grid(alpha=.3)
ax[1].plot(df.step, df.plan_grad_norm); ax[1].set_title('plan_grad_norm (signal through frozen executor)'); ax[1].grid(alpha=.3)
ax[2].plot(df.step, df.peakedness); ax[2].plot(df.step, df.tau); ax[2].legend(['peakedness','tau']); ax[2].set_title('gumbel anneal'); ax[2].grid(alpha=.3)
for a in ax: a.set_xlabel('step')
plt.tight_layout(); plt.show()